In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Analiza klasyfikacji jakości wina: Drzewa Decyzyjne i SVM\n",
    "\n",
    "## Wstęp\n",
    "W tym notatniku przeprowadzimy analizę klasyfikacji binarnej jakości wina na dwóch różnych zbiorach danych przy użyciu:\n",
    "1.  **Drzew Decyzyjnych (Decision Trees)**\n",
    "2.  **Maszyn Wektorów Nośnych (SVM - Support Vector Classifier)**\n",
    "\n",
    "Naszym celem jest przewidzenie, czy wino jest **\"Wysokiej Jakości\" (ocena >= 7)** na podstawie jego parametrów fizykochemicznych.\n",
    "\n",
    "### Zbiory danych\n",
    "1.  **White Wine Quality** (UCI/ML Mastery) - Zbiór dotyczący białego wina.\n",
    "2.  **WineQT** (Kaggle) - Zbiór dotyczący czerwonego wina.\n",
    "\n",
    "---\n",
    "### Import bibliotek"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Narzędzia do modelowania\n",
    "from sklearn.model_selection import train_test_split, GridSearchCV\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.tree import DecisionTreeClassifier\n",
    "from sklearn.svm import SVC\n",
    "from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay\n",
    "\n",
    "# Ustawienia wizualizacji\n",
    "sns.set(style=\"whitegrid\")\n",
    "%matplotlib inline\n",
    "\n",
    "print(\"Biblioteki zaimportowane.\")\n",
    "\n",
    "# Funkcja pomocnicza do tworzenia targetu binarnego\n",
    "# Definiujemy: 1 = Wysoka jakość (>= 7), 0 = Przeciętna/Słaba (< 7)\n",
    "def create_binary_target(quality_score):\n",
    "    return 1 if quality_score >= 7 else 0"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "--- \n",
    "# CZĘŚĆ 1: Zbiór danych White Wine Quality\n",
    "\n",
    "### 1.1. Ładowanie i przygotowanie danych"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Ścieżka do pliku w katalogu data/\n",
    "# UWAGA: Plik z UCI często używa średnika ';' jako separatora\n",
    "white_path = '../data/winequality-white.csv'\n",
    "\n",
    "try:\n",
    "    df1 = pd.read_csv(white_path, sep=';')\n",
    "    print(\"Zbiór White Wine załadowany.\")\n",
    "except FileNotFoundError:\n",
    "    print(f\"BŁĄD: Nie znaleziono pliku {white_path}.\")\n",
    "\n",
    "# Podgląd danych\n",
    "display(df1.head())\n",
    "\n",
    "# Transformacja do klasyfikacji binarnej\n",
    "df1['target'] = df1['quality'].apply(create_binary_target)\n",
    "\n",
    "print(\"\\nRozkład oryginalnych ocen jakości:\")\n",
    "print(df1['quality'].value_counts().sort_index())\n",
    "\n",
    "print(\"\\nRozkład nowej klasy binarnej 'target' (0: Przeciętne, 1: Wysoka jakość):\")\n",
    "print(df1['target'].value_counts(normalize=True))\n",
    "# Zbiór jest niezbalansowany (~21% win wysokiej jakości)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1.2. Wizualizacja danych (White Wine)\n",
    "Sprawdźmy wpływ poziomu alkoholu na binarną jakość wina."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(8, 6))\n",
    "sns.boxplot(x='target', y='alcohol', data=df1, palette='viridis')\n",
    "plt.title(\"Poziom alkoholu a jakość białego wina\")\n",
    "plt.xticks([0, 1], ['Przeciętne (<7)', 'Wysoka jakość (>=7)'])\n",
    "plt.ylabel(\"Alkohol (%)\")\n",
    "plt.show()\n",
    "# Wniosek: Wina wysokiej jakości mają tendencję do wyższej zawartości alkoholu."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1.3. Preprocessing i Trening Modeli (White Wine)\n",
    "Standaryzacja danych jest kluczowa dla SVM."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Przygotowanie cech (X) i targetu (y)\n",
    "X1 = df1.drop(['quality', 'target'], axis=1)\n",
    "y1 = df1['target']\n",
    "\n",
    "# Podział na zbiór treningowy i testowy ze stratyfikacją\n",
    "X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)\n",
    "\n",
    "# Skalowanie danych (dla SVM)\n",
    "scaler1 = StandardScaler()\n",
    "X1_train_scaled = scaler1.fit_transform(X1_train)\n",
    "X1_test_scaled = scaler1.transform(X1_test)\n",
    "\n",
    "# --- Funkcja do ewaluacji ---\n",
    "def evaluate_model(model, X_test, y_test, model_name):\n",
    "    y_pred = model.predict(X_test)\n",
    "    print(f\"--- Wyniki: {model_name} ---\")\n",
    "    # Skupiamy się na metrykach dla klasy 1 (Wysoka jakość), która jest mniejszościowa\n",
    "    print(classification_report(y_test, y_pred))\n",
    "\n",
    "# --- Model A: Drzewo Decyzyjne ---\n",
    "dt_model1 = DecisionTreeClassifier(random_state=42, max_depth=7)\n",
    "dt_model1.fit(X1_train, y1_train)\n",
    "evaluate_model(dt_model1, X1_test, y1_test, \"Drzewo Decyzyjne (White)\")\n",
    "\n",
    "# --- Model B: SVM (SVC) ---\n",
    "# Używamy class_weight='balanced' z powodu nierównowagi klas\n",
    "svm_model1 = SVC(kernel='rbf', class_weight='balanced', random_state=42)\n",
    "svm_model1.fit(X1_train_scaled, y1_train)\n",
    "evaluate_model(svm_model1, X1_test_scaled, y1_test, \"SVM RBF Balanced (White)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "--- \n",
    "# CZĘŚĆ 2: Zbiór danych WineQT (Czerwone Wino)\n",
    "\n",
    "Drugi, mniejszy zbiór danych dotyczący czerwonego wina.\n",
    "\n",
    "### 2.1. Ładowanie i przygotowanie danych"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "red_path = '../data/WineQT.csv'\n",
    "\n",
    "try:\n",
    "    df2 = pd.read_csv(red_path)\n",
    "    print(\"Zbiór WineQT (Red) załadowany.\")\n",
    "    # Usuwamy kolumnę 'Id', jeśli istnieje\n",
    "    if 'Id' in df2.columns:\n",
    "        df2 = df2.drop('Id', axis=1)\n",
    "except FileNotFoundError:\n",
    "    print(f\"BŁĄD: Nie znaleziono pliku {red_path}.\")\n",
    "\n",
    "display(df2.head())\n",
    "\n",
    "# Transformacja do klasyfikacji binarnej (ta sama logika: >=7 to wysoka jakość)\n",
    "df2['target'] = df2['quality'].apply(create_binary_target)\n",
    "\n",
    "print(\"\\nRozkład klasy binarnej 'target' (WineQT):\")\n",
    "print(df2['target'].value_counts(normalize=True))\n",
    "# Ten zbiór jest jeszcze bardziej niezbalansowany (~13.5% wysokiej jakości)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2.2. Wizualizacja (WineQT)\n",
    "Sprawdźmy kwasowość lotną (volatile acidity) - często uważaną za wadę wina w wysokich stężeniach."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(8, 6))\n",
    "sns.boxplot(x='target', y='volatile acidity', data=df2, palette='magma')\n",
    "plt.title(\"Kwasowość lotna a jakość czerwonego wina\")\n",
    "plt.xticks([0, 1], ['Przeciętne (<7)', 'Wysoka jakość (>=7)'])\n",
    "plt.ylabel(\"Volatile Acidity\")\n",
    "plt.show()\n",
    "# Wniosek: Wina wysokiej jakości mają wyraźnie niższą kwasowość lotną."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2.3. Preprocessing i Trening Modeli (WineQT)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X2 = df2.drop(['quality', 'target'], axis=1)\n",
    "y2 = df2['target']\n",
    "\n",
    "X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)\n",
    "\n",
    "scaler2 = StandardScaler()\n",
    "X2_train_scaled = scaler2.fit_transform(X2_train)\n",
    "X2_test_scaled = scaler2.transform(X2_test)\n",
    "\n",
    "# --- Model A: Drzewo Decyzyjne ---\n",
    "dt_model2 = DecisionTreeClassifier(random_state=42, max_depth=6)\n",
    "dt_model2.fit(X2_train, y2_train)\n",
    "evaluate_model(dt_model2, X2_test, y2_test, \"Drzewo Decyzyjne (WineQT)\")\n",
    "\n",
    "# --- Model B: SVM (SVC) ---\n",
    "svm_model2 = SVC(kernel='rbf', class_weight='balanced', random_state=42)\n",
    "svm_model2.fit(X2_train_scaled, y2_train)\n",
    "evaluate_model(svm_model2, X2_test_scaled, y2_test, \"SVM RBF Balanced (WineQT)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "--- \n",
    "# CZĘŚĆ 3: Analiza funkcji jądra (Kernel Function) w SVM\n",
    "\n",
    "Przeprowadzimy analizę wpływu różnych kerneli i hiperparametrów na jakość klasyfikacji, używając **większego zbioru danych (White Wine)** i techniki `GridSearchCV`.\n",
    "\n",
    "Badane parametry:\n",
    "* **Kernel**: `linear`, `poly`, `rbf`, `sigmoid`.\n",
    "* **C**: Parametr regularyzacji.\n",
    "* **Gamma**: Współczynnik jądra (dla rbf, poly, sigmoid).\n",
    "* **Degree**: Stopień wielomianu (dla poly)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"Rozpoczynam analizę kerneli SVM (GridSearchCV) na zbiorze White Wine...\")\n",
    "# Używamy mniejszego podzbioru treningowego, żeby przyspieszyć obliczenia w demonstracji\n",
    "# W realnym scenariuszu użylibyśmy całego X1_train_scaled\n",
    "X_grid, _, y_grid, _ = train_test_split(X1_train_scaled, y1_train, train_size=1000, random_state=42, stratify=y1_train)\n",
    "\n",
    "# Siatka parametrów\n",
    "param_grid = [\n",
    "    {'kernel': ['linear'], 'C': [1, 10]},\n",
    "    {'kernel': ['rbf'], 'C': [1, 10, 100], 'gamma': ['scale', 0.1]},\n",
    "    {'kernel': ['poly'], 'degree': [2, 3], 'C': [1], 'gamma': ['scale']},\n",
    "    {'kernel': ['sigmoid'], 'C': [1], 'gamma': ['scale']}\n",
    "]\n",
    "\n",
    "# GridSearchCV (używamy f1-score dla klasy mniejszościowej jako metryki, bo zbiór jest niezbalansowany)\n",
    "grid_search = GridSearchCV(SVC(class_weight='balanced', random_state=42), \n",
    "                           param_grid, \n",
    "                           cv=3, \n",
    "                           scoring='f1', \n",
    "                           verbose=1, n_jobs=-1)\n",
    "\n",
    "grid_search.fit(X_grid, y_grid)\n",
    "\n",
    "print(\"\\nGridSearch zakończony.\")\n",
    "print(\"Najlepsze parametry:\", grid_search.best_params_)\n",
    "print(f\"Najlepszy wynik F1 (CV): {grid_search.best_score_:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Analiza wyników GridSearch (White Wine)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "results_df = pd.DataFrame(grid_search.cv_results_)\n",
    "relevant_cols = ['param_kernel', 'param_C', 'param_gamma', 'param_degree', 'mean_test_score', 'rank_test_score']\n",
    "display(results_df[relevant_cols].sort_values(by='rank_test_score').head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ewaluacja najlepszego modelu na zbiorze testowym"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "best_svm = grid_search.best_estimator_\n",
    "\n",
    "print(f\"\\nEwaluacja najlepszego modelu ({grid_search.best_params_['kernel']}) na PEŁNYM zbiorze testowym White Wine:\")\n",
    "final_pred = best_svm.predict(X1_test_scaled)\n",
    "\n",
    "print(classification_report(y1_test, final_pred))\n",
    "\n",
    "cm = confusion_matrix(y1_test, final_pred)\n",
    "disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Przeciętne', 'Wysoka Jakość'])\n",
    "disp.plot(cmap='YlGnBu')\n",
    "plt.grid(False)\n",
    "plt.title(\"Macierz Pomyłek - Najlepszy SVM\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Podsumowanie\n",
    "Szczegółowe podsumowanie analizy kerneli znajduje się w pliku `reports/svm_kernel_summary_white_wine.md`."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.13"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}